# Notebook 2: Missingness Injection and Verification

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import config
from src.data.loader import load_network, sample_data
from src.data.missingness import inject_missingness

## 1. Load Sachs data

In [ ]:
network = load_network('sachs')
df = sample_data(network)
print(f"Shape: {df.shape}")
df.head()

## 2. Inject missingness and verify rates

In [ ]:
verification = []

for mechanism in config.MISSINGNESS_MECHANISMS:
    for rate in config.MISSINGNESS_LEVELS:
        df_missing = inject_missingness(df, mechanism, rate)
        actual_rate = df_missing.isna().mean().mean()
        verification.append({
            "mechanism": mechanism,
            "target_rate": rate,
            "actual_rate": round(actual_rate, 4),
            "rows_complete": int(df_missing.dropna().shape[0]),
        })

pd.DataFrame(verification)

## 3. Visualise missingness pattern heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, mechanism in zip(axes, ['MCAR', 'MNAR']):
    df_missing = inject_missingness(df, mechanism, 0.30)
    sns.heatmap(
        df_missing.head(100).isna().astype(int),
        ax=ax,
        cbar=False,
        cmap='Greys',
        yticklabels=False,
    )
    ax.set_title(f'{mechanism} — 30% missingness (first 100 rows)')
    ax.set_xlabel('Variable')

plt.tight_layout()
plt.savefig('../results/missingness_heatmaps.png', dpi=150)
plt.show()
print("Saved: results/missingness_heatmaps.png")

## 4. Check: MNAR concentrates missingness on the most common value

In [ ]:
col = df.columns[0]
mode_val = df[col].mode()[0]

df_mnar = inject_missingness(df, 'MNAR', 0.30)

# check NaN rate among mode vs non-mode rows
mode_mask = df[col] == mode_val
nan_among_mode = df_mnar.loc[mode_mask, col].isna().mean()
nan_among_nonmode = df_mnar.loc[~mode_mask, col].isna().mean()

print(f"Column: {col}, mode value: {mode_val}")
print(f"NaN rate among mode-value rows:    {nan_among_mode:.2%}")
print(f"NaN rate among non-mode-value rows: {nan_among_nonmode:.2%}")
print("\nMNAR confirmed if first rate >> second rate.")